# Leakage Stress Testing for Financial Fraud Detection

This notebook runs the first novelty experiment for the revised paper: leakage and feature-availability stress testing.

It is designed for Google Colab with a T4 GPU, but most tree models still run on CPU. The notebook is resumable: after each dataset/scenario/model result, it writes a CSV checkpoint to Google Drive. If Colab disconnects, rerun the notebook and completed jobs will be skipped.

Primary outputs:

- strict safe-pipeline results
- unsafe preprocessing stress results
- unsafe SMOTE-before-split results
- D3 ledger-state stress results
- Leakage Inflation Index tables
- 600-DPI journal/conference figures

## Experiment Design

Main protocol:

1. **SAFE_STRICT**: split first, fit preprocessing on training only, apply SMOTE only to training data when the scenario requests it.
2. **UNSAFE_PREPROCESS_FULL_FIT**: fit preprocessing on the full dataset before splitting. This tests how much full-data transformation can inflate estimates.
3. **UNSAFE_SMOTE_BEFORE_SPLIT**: fit preprocessing on the full dataset, apply SMOTE to the full transformed matrix, then split. This is deliberately unsafe and should never be used as the main result.
4. **D3_LEDGER_STATE_STRESS**: compare D3 strict pre-transaction features against a ledger-state feature set that adds `newbalanceOrig` and `newbalanceDest`.

Main metric:

`Leakage Inflation Index = stress_metric - safe_strict_metric`

For cost, lower is better, so the cost inflation interpretation is handled separately.

In [ ]:
# Colab setup
!pip -q install -U imbalanced-learn lightgbm xgboost seaborn pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

from pathlib import Path

# Change this once you provide the exact Google Drive folder.
# Your Drive screenshot shows: My Drive > Credit__ > Datasets
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

# The notebook will search these subfolders for creditcard.csv, fraudTest.csv, and PS.csv.
DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / 'Datasets',
    PROJECT_ROOT / '00_source_datasets',
    PROJECT_ROOT,
]

RUN_NAME = 'leakage_stress_v1'
EXPERIMENT_ROOT = PROJECT_ROOT / '03_leakage_stress_testing' / RUN_NAME
CHECKPOINT_DIR = EXPERIMENT_ROOT / 'checkpoints'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
LOGS_DIR = EXPERIMENT_ROOT / 'logs'

for d in [EXPERIMENT_ROOT, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Runtime controls. Start with these; increase after the first successful run.
RANDOM_SEED = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

# Caps keep Colab T4 sessions realistic. D1 is deduplicated full data.
CAPS = {
    'D1_creditcard': None,
    'D2_fraudTest': 300_000,
    'D3_PaySim': 200_000,
}

# Use fewer rows for a smoke test. Set to False for paper-grade run.
SMOKE_TEST = False
SMOKE_CAP = 30_000

# Models to run for leakage stress.
# LogisticRegression is fast and helps detect leakage inflation.
# XGBoost/LightGBM are main tree baselines.
MODEL_NAMES = ['logistic_regression', 'xgboost', 'lightgbm']

# Scenario list. Keep this as-is for Milestone 03.
SCENARIOS = [
    'SAFE_STRICT',
    'UNSAFE_PREPROCESS_FULL_FIT',
    'UNSAFE_SMOTE_BEFORE_SPLIT',
    'D3_STRICT_SAFE',
    'D3_LEDGER_STATE_SAFE',
]

print('Project root:', PROJECT_ROOT)
print('Experiment root:', EXPERIMENT_ROOT)

In [ ]:
import os
import json
import time
import math
import shutil
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)

try:
    import xgboost as xgb
except Exception as exc:
    xgb = None
    print('XGBoost unavailable:', exc)

try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print('LightGBM unavailable:', exc)

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

def now():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def write_csv_atomic(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'run.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def checkpoint_path(dataset, scenario, model_name):
    safe = f'{dataset}__{scenario}__{model_name}.csv'
    return CHECKPOINT_DIR / safe

def is_done(dataset, scenario, model_name):
    return checkpoint_path(dataset, scenario, model_name).exists()

In [ ]:
def find_dataset_file(file_name):
    candidates = []
    for directory in DATASET_SEARCH_DIRS:
        candidates.append(directory / file_name)
        candidates.extend(directory.glob(f'**/{file_name}') if directory.exists() else [])
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f'Could not find {file_name}. Checked: ' + ', '.join(str(d) for d in DATASET_SEARCH_DIRS)
    )

DATASET_FILES = {
    'D1_creditcard': find_dataset_file('creditcard.csv'),
    'D2_fraudTest': find_dataset_file('fraudTest.csv'),
    'D3_PaySim': find_dataset_file('PS.csv'),
}

for name, path in DATASET_FILES.items():
    append_log(f'{name}: {path}')

write_json(EXPERIMENT_ROOT / 'dataset_file_manifest.json', {k: str(v) for k, v in DATASET_FILES.items()})

In [ ]:
def cap_by_retaining_fraud(df, label_col, cap, seed=RANDOM_SEED):
    if cap is None or len(df) <= cap:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    y = pd.to_numeric(df[label_col], errors='coerce')
    fraud = df[y == 1]
    legit = df[y == 0]
    legit_n = max(cap - len(fraud), 0)
    legit_sample = legit.sample(n=min(legit_n, len(legit)), random_state=seed)
    out = pd.concat([fraud, legit_sample], axis=0)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

def load_dataset(dataset_name):
    path = DATASET_FILES[dataset_name]
    append_log(f'Loading {dataset_name}')
    df = pd.read_csv(path, low_memory=False)

    if dataset_name == 'D1_creditcard':
        df = df.drop_duplicates().reset_index(drop=True)
        label_col = 'Class'

    elif dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'

    elif dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        df = df.dropna(subset=[label_col]).reset_index(drop=True)
        df[label_col] = df[label_col].astype(int)

    else:
        raise ValueError(dataset_name)

    cap = SMOKE_CAP if SMOKE_TEST else CAPS[dataset_name]
    df = cap_by_retaining_fraud(df, label_col, cap, RANDOM_SEED)

    fraud = int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum())
    manifest = {
        'dataset': dataset_name,
        'rows_after_cleaning_and_cap': int(len(df)),
        'columns': int(df.shape[1]),
        'label_col': label_col,
        'fraud_rows': fraud,
        'fraud_rate_pct': float(fraud / len(df) * 100),
        'cap': cap,
        'smoke_test': SMOKE_TEST,
    }
    write_json(EXPERIMENT_ROOT / f'{dataset_name}_load_manifest.json', manifest)
    append_log(f"{dataset_name}: rows={len(df):,}, fraud={fraud:,}, fraud_rate={manifest['fraud_rate_pct']:.4f}%")
    return df, label_col

In [ ]:
def prepare_features(df, dataset_name, feature_policy):
    df = df.copy()

    if dataset_name == 'D1_creditcard':
        label_col = 'Class'
        keep = [f'V{i}' for i in range(1, 29)] + ['Amount']
        # Time is reserved for chronological validation, not strict random-split input.
        X = df[keep].copy()
        y = df[label_col].astype(int)
        time_values = df['Time'].copy()
        return X, y, time_values

    if dataset_name == 'D2_fraudTest':
        label_col = 'is_fraud'
        # Strict protocol removes direct identifiers and raw timestamps.
        df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        df['dob_dt'] = pd.to_datetime(df['dob'], errors='coerce')
        df['trans_hour'] = df['trans_dt'].dt.hour
        df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek
        df['trans_month'] = df['trans_dt'].dt.month
        df['age_years'] = (df['trans_dt'] - df['dob_dt']).dt.days / 365.25

        keep = [
            'category', 'amt', 'state', 'city_pop',
            'trans_hour', 'trans_dayofweek', 'trans_month', 'age_years'
        ]
        # Gender/job/merchant/location-rich fields are intentionally not in strict v1.
        X = df[keep].copy()
        y = df[label_col].astype(int)
        time_values = pd.to_datetime(df['trans_date_trans_time'], errors='coerce')
        return X, y, time_values

    if dataset_name == 'D3_PaySim':
        label_col = 'isFraud'
        keep = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
        if feature_policy == 'ledger_state':
            keep += ['newbalanceOrig', 'newbalanceDest']
        X = df[keep].copy()
        y = df[label_col].astype(int)
        time_values = df['step'].copy()
        return X, y, time_values

    raise ValueError(dataset_name)

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler(with_centering=False)),
    ])

    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ])
    return preprocessor

In [ ]:
def build_model(model_name, scale_pos_weight=1.0):
    if model_name == 'logistic_regression':
        return LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            solver='saga',
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )

    if model_name == 'xgboost':
        if xgb is None:
            raise RuntimeError('XGBoost not available')
        # device='cuda' uses GPU when available, with fallback handled by XGBoost warnings.
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective='binary:logistic',
            eval_metric='aucpr',
            tree_method='hist',
            device='cuda',
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )

    if model_name == 'lightgbm':
        if lgb is None:
            raise RuntimeError('LightGBM not available')
        return lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            class_weight='balanced',
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1,
        )

    raise ValueError(model_name)

def get_scores(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    raise RuntimeError('Model has no score method')

def choose_threshold(y_val, scores_val, c_fn=100, c_fp=1):
    thresholds = np.round(np.arange(0.01, 0.91, 0.01), 2)
    best = None
    for tau in thresholds:
        pred = (scores_val >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        cost = c_fp * fp + c_fn * fn
        mcc = matthews_corrcoef(y_val, pred) if len(np.unique(pred)) > 1 else 0.0
        candidate = (cost, fn, fp, -mcc, tau)
        if best is None or candidate < best:
            best = candidate
    return float(best[-1])

def evaluate_predictions(y_true, scores, tau):
    pred = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = len(y_true)
    out = {
        'threshold': float(tau),
        'n_test': int(n),
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, pred)) if len(np.unique(pred)) > 1 else 0.0,
        'average_precision': float(average_precision_score(y_true, scores)),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': float(brier_score_loss(y_true, scores)),
        'false_alerts_per_10k': float(fp / n * 10_000),
        'missed_frauds_per_10k': float(fn / n * 10_000),
        'cost_per_10k': float((fp + 100 * fn) / n * 10_000),
    }
    return out

In [ ]:
def split_indices(X, y):
    idx = np.arange(len(y))
    train_val_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
    )
    y_train_val = y.iloc[train_val_idx] if hasattr(y, 'iloc') else y[train_val_idx]
    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=VALIDATION_SIZE,
        stratify=y_train_val,
        random_state=RANDOM_SEED,
    )
    return train_idx, val_idx, test_idx

def run_safe_pipeline(X, y, model_name, use_smote=False):
    train_idx, val_idx, test_idx = split_indices(X, y)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    preprocessor = make_preprocessor(X_train)
    X_train_t = preprocessor.fit_transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)

    if use_smote:
        smote = SMOTE(k_neighbors=5, random_state=RANDOM_SEED)
        X_train_t, y_train = smote.fit_resample(X_train_t, y_train)

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    return metrics

def run_unsafe_preprocess_full_fit(X, y, model_name):
    train_idx, val_idx, test_idx = split_indices(X, y)
    preprocessor = make_preprocessor(X)
    X_all_t = preprocessor.fit_transform(X)

    X_train_t, y_train = X_all_t[train_idx], y.iloc[train_idx]
    X_val_t, y_val = X_all_t[val_idx], y.iloc[val_idx]
    X_test_t, y_test = X_all_t[test_idx], y.iloc[test_idx]

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    return metrics

def run_unsafe_smote_before_split(X, y, model_name):
    preprocessor = make_preprocessor(X)
    X_all_t = preprocessor.fit_transform(X)
    smote = SMOTE(k_neighbors=5, random_state=RANDOM_SEED)
    X_res, y_res = smote.fit_resample(X_all_t, y)

    train_idx, val_idx, test_idx = split_indices(pd.DataFrame({'i': np.arange(len(y_res))}), pd.Series(y_res))
    X_train_t, y_train = X_res[train_idx], pd.Series(y_res).iloc[train_idx]
    X_val_t, y_val = X_res[val_idx], pd.Series(y_res).iloc[val_idx]
    X_test_t, y_test = X_res[test_idx], pd.Series(y_res).iloc[test_idx]

    pos = max(int(np.sum(y_train == 1)), 1)
    neg = max(int(np.sum(y_train == 0)), 1)
    model = build_model(model_name, scale_pos_weight=neg / pos)
    t0 = time.time()
    model.fit(X_train_t, y_train)
    fit_seconds = time.time() - t0

    val_scores = get_scores(model, X_val_t)
    tau = choose_threshold(y_val, val_scores)
    test_scores = get_scores(model, X_test_t)
    metrics = evaluate_predictions(y_test, test_scores, tau)
    metrics['fit_seconds'] = float(fit_seconds)
    metrics['unsafe_resampled_rows'] = int(X_res.shape[0])
    return metrics

In [ ]:
def run_one(dataset_name, scenario, model_name):
    if is_done(dataset_name, scenario, model_name):
        append_log(f'SKIP existing checkpoint: {dataset_name} / {scenario} / {model_name}')
        return pd.read_csv(checkpoint_path(dataset_name, scenario, model_name)).iloc[0].to_dict()

    if scenario.startswith('D3') and dataset_name != 'D3_PaySim':
        return None
    if dataset_name == 'D3_PaySim' and scenario in ['SAFE_STRICT', 'UNSAFE_PREPROCESS_FULL_FIT', 'UNSAFE_SMOTE_BEFORE_SPLIT']:
        # D3 strict safe is labeled separately so we can compare with ledger-state stress clearly.
        return None

    df, label_col = load_dataset(dataset_name)

    if scenario == 'D3_LEDGER_STATE_SAFE':
        X, y, time_values = prepare_features(df, dataset_name, feature_policy='ledger_state')
        metrics = run_safe_pipeline(X, y, model_name, use_smote=False)
        feature_policy = 'ledger_state'
        scenario_type = 'safe_pipeline_with_post_transaction_balances'

    elif scenario == 'D3_STRICT_SAFE':
        X, y, time_values = prepare_features(df, dataset_name, feature_policy='strict')
        metrics = run_safe_pipeline(X, y, model_name, use_smote=False)
        feature_policy = 'strict'
        scenario_type = 'safe_pipeline_strict_features'

    else:
        X, y, time_values = prepare_features(df, dataset_name, feature_policy='strict')
        feature_policy = 'strict'
        if scenario == 'SAFE_STRICT':
            metrics = run_safe_pipeline(X, y, model_name, use_smote=False)
            scenario_type = 'safe_pipeline_strict_features'
        elif scenario == 'UNSAFE_PREPROCESS_FULL_FIT':
            metrics = run_unsafe_preprocess_full_fit(X, y, model_name)
            scenario_type = 'unsafe_preprocessing_full_data_fit'
        elif scenario == 'UNSAFE_SMOTE_BEFORE_SPLIT':
            metrics = run_unsafe_smote_before_split(X, y, model_name)
            scenario_type = 'unsafe_smote_before_split'
        else:
            raise ValueError(scenario)

    row = {
        'run_name': RUN_NAME,
        'timestamp': now(),
        'dataset': dataset_name,
        'scenario': scenario,
        'scenario_type': scenario_type,
        'model': model_name,
        'feature_policy': feature_policy,
        'random_seed': RANDOM_SEED,
        'smoke_test': SMOKE_TEST,
        'rows_used': int(len(df)),
        'fraud_rows_used': int((pd.to_numeric(df[label_col], errors='coerce') == 1).sum()),
        **metrics,
    }
    out = pd.DataFrame([row])
    write_csv_atomic(checkpoint_path(dataset_name, scenario, model_name), out)
    append_log(f'DONE {dataset_name} / {scenario} / {model_name}: AP={row["average_precision"]:.4f}, MCC={row["mcc"]:.4f}')
    return row

all_rows = []
for dataset_name in ['D1_creditcard', 'D2_fraudTest', 'D3_PaySim']:
    for scenario in SCENARIOS:
        for model_name in MODEL_NAMES:
            try:
                row = run_one(dataset_name, scenario, model_name)
                if row is not None:
                    all_rows.append(row)
                    write_csv_atomic(RESULTS_DIR / 'partial_results_latest.csv', pd.DataFrame(all_rows))
            except Exception as exc:
                append_log(f'ERROR {dataset_name} / {scenario} / {model_name}: {repr(exc)}')
                error_path = CHECKPOINT_DIR / f'ERROR__{dataset_name}__{scenario}__{model_name}.json'
                write_json(error_path, {
                    'dataset': dataset_name,
                    'scenario': scenario,
                    'model': model_name,
                    'error': repr(exc),
                    'timestamp': now(),
                })

checkpoint_files = sorted(CHECKPOINT_DIR.glob('*.csv'))
result_frames = [pd.read_csv(p) for p in checkpoint_files]
results = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'leakage_stress_results.csv', results)
results

In [ ]:
results_path = RESULTS_DIR / 'leakage_stress_results.csv'
results = pd.read_csv(results_path)

safe_refs = []
for _, row in results.iterrows():
    if row['dataset'] in ['D1_creditcard', 'D2_fraudTest']:
        safe_scenario = 'SAFE_STRICT'
    elif row['dataset'] == 'D3_PaySim':
        safe_scenario = 'D3_STRICT_SAFE'
    else:
        continue
    safe_refs.append((row['dataset'], row['model'], safe_scenario))

safe = results[results['scenario'].isin(['SAFE_STRICT', 'D3_STRICT_SAFE'])].copy()
safe = safe[['dataset', 'model', 'average_precision', 'mcc', 'recall', 'precision', 'cost_per_10k', 'brier']]
safe = safe.rename(columns={
    'average_precision': 'safe_average_precision',
    'mcc': 'safe_mcc',
    'recall': 'safe_recall',
    'precision': 'safe_precision',
    'cost_per_10k': 'safe_cost_per_10k',
    'brier': 'safe_brier',
})

inflation = results.merge(safe, on=['dataset', 'model'], how='left')
inflation['lii_average_precision'] = inflation['average_precision'] - inflation['safe_average_precision']
inflation['lii_mcc'] = inflation['mcc'] - inflation['safe_mcc']
inflation['lii_recall'] = inflation['recall'] - inflation['safe_recall']
inflation['lii_precision'] = inflation['precision'] - inflation['safe_precision']
inflation['cost_delta_per_10k'] = inflation['cost_per_10k'] - inflation['safe_cost_per_10k']
inflation['brier_delta'] = inflation['brier'] - inflation['safe_brier']

write_csv_atomic(RESULTS_DIR / 'leakage_inflation_index.csv', inflation)
inflation[['dataset','scenario','model','average_precision','safe_average_precision','lii_average_precision','mcc','safe_mcc','lii_mcc','cost_per_10k','safe_cost_per_10k','cost_delta_per_10k']]

In [ ]:
inflation = pd.read_csv(RESULTS_DIR / 'leakage_inflation_index.csv')

plot_df = inflation[~inflation['scenario'].isin(['SAFE_STRICT', 'D3_STRICT_SAFE'])].copy()

for metric, ylabel in [
    ('lii_average_precision', 'Leakage Inflation Index: Average Precision'),
    ('lii_mcc', 'Leakage Inflation Index: MCC'),
    ('cost_delta_per_10k', 'Cost delta per 10,000 transactions'),
]:
    plt.figure(figsize=(8.2, 4.8))
    ax = sns.barplot(
        data=plot_df,
        x='dataset',
        y=metric,
        hue='scenario',
        errorbar=None,
    )
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' by Dataset and Stress Scenario')
    ax.legend(title='Scenario', loc='best', frameon=True)
    plt.xticks(rotation=0)
    plt.tight_layout()
    png = FIGURES_DIR / f'{metric}_by_dataset.png'
    pdf = FIGURES_DIR / f'{metric}_by_dataset.pdf'
    plt.savefig(png, dpi=600, bbox_inches='tight')
    plt.savefig(pdf, bbox_inches='tight')
    plt.show()
    append_log(f'Saved figure: {png}')

# Heatmap for AP inflation by dataset/model/scenario.
heat = plot_df.pivot_table(
    index=['dataset', 'model'],
    columns='scenario',
    values='lii_average_precision',
    aggfunc='mean',
)
plt.figure(figsize=(9, max(3.5, 0.45 * len(heat))))
ax = sns.heatmap(heat, annot=True, fmt='.3f', cmap='vlag', center=0, linewidths=0.4)
ax.set_title('Average Precision Leakage Inflation Index')
ax.set_xlabel('Stress scenario')
ax.set_ylabel('Dataset / model')
plt.tight_layout()
png = FIGURES_DIR / 'ap_leakage_inflation_heatmap.png'
pdf = FIGURES_DIR / 'ap_leakage_inflation_heatmap.pdf'
plt.savefig(png, dpi=600, bbox_inches='tight')
plt.savefig(pdf, bbox_inches='tight')
plt.show()
append_log(f'Saved figure: {png}')

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'experiment_root': str(EXPERIMENT_ROOT),
    'results_file': str(RESULTS_DIR / 'leakage_stress_results.csv'),
    'inflation_file': str(RESULTS_DIR / 'leakage_inflation_index.csv'),
    'figures_dir': str(FIGURES_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'n_result_rows': int(len(pd.read_csv(RESULTS_DIR / 'leakage_stress_results.csv'))),
}
write_json(EXPERIMENT_ROOT / 'run_summary.json', summary)
summary

## Paper-Ready Interpretation Template

After the run completes, use `results/leakage_inflation_index.csv` to write the paper result.

Suggested wording:

> We define a Leakage Inflation Index (LII) as the difference between the metric obtained under a stress condition and the corresponding strict safe-pipeline metric for the same dataset and model. Positive LII for discrimination metrics indicates that the stress condition improved the apparent result relative to the strict protocol. For cost, negative values indicate lower apparent operating cost, but this improvement is not valid when produced by an unsafe evaluation design.

Report:

- largest AP inflation
- largest MCC inflation
- whether unsafe SMOTE-before-split produces suspiciously large gains
- whether D3 post-transaction balances materially change the result
- whether the strict feature governance changes conclusions compared with the earlier manuscript